# Baseline — Tricy Table

**Competition:** a tabular regression task with **missing values and hidden patterns**
planted in the features.

- **Task:** predict `target` for every test row
- **Metric:** `score = 1 / (1 + MAE)` — higher is better
- **Kaggle link:** _TODO: add link_

**Approach:** LightGBM handles missing values natively and finds interactions on its
own — a strong first move for any tabular problem.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

DATA_DIR = "."
train = pd.read_csv(f"{DATA_DIR}/train_tables.csv")
test  = pd.read_csv(f"{DATA_DIR}/test_tables.csv")
print(train.shape, test.shape)
train.head(3)

(28492, 14) (12212, 13)


,id,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,day,hour,minute,target
0,train_00000,281.646409,1533.835429,20.502419,73.773726,2934.296784,NaN,NaN,1152.884069,547.505072,2.0,16.0,4.0,262.062136
1,train_00001,362.690848,1978.855064,20.423763,78.647682,2995.408503,44.379741,1913.907575,1249.216761,641.077826,2.0,8.0,46.0,501.775741
2,train_00002,450.636690,2656.533713,20.463104,86.110627,2874.163323,NaN,2641.484158,1330.960413,642.181584,2.0,NaN,50.0,745.107882


In [2]:
target_col = "target"
id_col = "id" if "id" in test.columns else test.columns[0]
features = [c for c in train.columns if c not in {target_col, id_col}]
# encode any text columns as categories
for c in features:
    if train[c].dtype == object:
        cats = pd.concat([train[c], test[c]]).astype("category").cat.categories
        train[c] = pd.Categorical(train[c], categories=cats).codes
        test[c]  = pd.Categorical(test[c],  categories=cats).codes
X, y, Xt = train[features], train[target_col].values, test[features]
print(len(features), "features")

12 features


In [3]:
oof = np.zeros(len(train)); preds = np.zeros(len(test))
for tr_idx, va_idx in KFold(5, shuffle=True, random_state=0).split(X):
    m = lgb.LGBMRegressor(n_estimators=1200, learning_rate=0.05, num_leaves=63,
                          random_state=0, verbose=-1)
    m.fit(X.iloc[tr_idx], y[tr_idx])
    oof[va_idx] = m.predict(X.iloc[va_idx])
    preds += m.predict(Xt) / 5
mae = mean_absolute_error(y, oof)
print(f"CV MAE: {mae:.4f}   score: {1/(1+mae):.5f}")

CV MAE: 3.3717   score: 0.22874


In [4]:
sub = pd.DataFrame({id_col: test[id_col], "target": preds})
sub.to_csv("submission.csv", index=False)
sub.head()

,id,target
0,test_00000,732.910540
1,test_00001,681.722151
2,test_00002,681.065076
3,test_00003,685.144761
4,test_00004,683.546838


## Ideas to improve

- Inspect feature importances — "tricy" tables usually hide a few *golden* engineered
  features (ratios, sums, parity of some column...). Plot target vs top features.
- Check whether missingness itself is informative (`isna()` indicator features).
- Try target transformations (log) if the target is skewed, and model ensembles.
